In [1]:
from pathlib import Path
import os
import time
import numpy as np
from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter
import re
import json
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

d:\ASUS\software\Anaconda3\envs\rag_agent\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
d:\ASUS\software\Anaconda3\envs\rag_agent\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [39]:
# 定义输入输出路径
INPUT_DIR = Path("input")
os.makedirs(INPUT_DIR, exist_ok=True)
OUTPUT_DIR = Path("output")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"输入目录：{INPUT_DIR.resolve()}")
print(f"输出目录：{OUTPUT_DIR.resolve()}")

输入目录：D:\workspace\new-rag\python_backend\python_services\test\input
输出目录：D:\workspace\new-rag\python_backend\python_services\test\output


In [40]:
# 接入mineru、模型
MINERU_BASE_URL = "https://mineru.net"
MINERU_API_KEY = os.getenv("MINERU_API_KEY")
if not MINERU_API_KEY:
    raise ValueError("MINERU_API_KEY 环境变量未设置")

LITELLM_BASE_URL = "http://localhost:4000"
LITELLM_API_KEY = os.getenv("LITELLM_API_KEY")
if not LITELLM_API_KEY:
    raise ValueError("LITELLM_API_KEY 环境变量未设置")
EMBEDDING_MODEL = "github_copilot/text-embedding-ada-002"
DEFAULT_MODEL = "gpt-4o"

print(f"模型：{DEFAULT_MODEL}, 嵌入模型：{EMBEDDING_MODEL}")

模型：gpt-4o, 嵌入模型：github_copilot/text-embedding-ada-002


In [41]:
from pydantic import BaseModel


class StoredData(BaseModel):
    id: str
    chunk: str
    sub_questions: list[str]
    subq_embeddings: list[list[float]]
    summary: str
    summary_embedding: list[float]
    metadata: dict

print("StoredData模型已定义(用于后续存储数据结构)")

StoredData模型已定义(用于后续存储数据结构)


In [42]:
# 进行PDF文档解析
import requests
import time
import zipfile
import io
import uuid

pdf_name = "video.pdf"
input_file = INPUT_DIR / pdf_name
if not input_file.exists():
    raise FileNotFoundError(f"输入文件 {input_file.resolve()} 不存在")

print(f"pdf文件已获取，大小：{input_file.stat().st_size / 1024:.2f} KB")

md_output_dir = OUTPUT_DIR / input_file.stem
md_output_dir.mkdir(parents=True, exist_ok=True)
md_output_path = md_output_dir / "extracted.md"

pdf文件已获取，大小：544.87 KB


In [43]:
def parsePDF():
    # 1. 获取上传URL
    headers = {
        "Authorization": f"Bearer {MINERU_API_KEY}",
        "Content-Type": "application/json"
    }

    print("正在获取上传URL...")
    response = requests.post(
        url=f"{MINERU_BASE_URL}/api/v4/file-urls/batch",
        headers=headers,
        json={
            "files": [{"name": pdf_name, "data_id": pdf_name}],
            "model_version": "vlm",
        }
    )
    response.raise_for_status()

    data = response.json()["data"]
    batch_id = data["batch_id"]
    file_urls = data["file_urls"]

    print(f"获取上传URL成功，batch_id: {batch_id}")

    # 2. 上传PDF文件到获取的URL
    print("正在上传PDF文件到MinerU的URL中...")
    with open(input_file, "rb") as f:
        file_data = f.read()
        upload_response = requests.put(file_urls[0], data=file_data)
        upload_response.raise_for_status()

    print("PDF文件上传成功！")

    # 3. 轮训检查处理状态
    print("正在轮询检查处理状态...")

    max_wait = 600  # 最长等待10分钟
    poll_interval = 3  # 每3秒检查一次
    elapsed = 0
    full_zip_url = None

    while elapsed < max_wait:
        status_response = requests.get(
            f"{MINERU_BASE_URL}/api/v4/extract-results/batch/{batch_id}",
            headers={"Authorization": f"Bearer {MINERU_API_KEY}"}
        )

        result = status_response.json()["data"]["extract_result"][0]
        state = result["state"]
        if state == "done":
            full_zip_url = result["full_zip_url"]
            print(f"解析完成，下载链接：{full_zip_url}")
            break
        elif state == "failed":
            error_message = result.get("err_msg", "未知错误")
            print(f"解析失败：{error_message}")
            break
        elif state == "running":
            progress = result.get("extract_progress", {})
            extracted = progress.get("extracted_pages", 0)
            total = progress.get("total_pages", "?")
            print(f"   {extracted}/{total} 页 (已等待 {elapsed}秒)", end="\r")

        time.sleep(poll_interval)
        elapsed += poll_interval

    if elapsed >= max_wait:
        print("等待超时，未能完成解析")

    # 4. 下载解析结果的zip文件，并解压
    if full_zip_url:
        print("正在下载解析结果的zip文件，并解压...")
        zip_response = requests.get(full_zip_url)

        with zipfile.ZipFile(io.BytesIO(zip_response.content)) as zf:
            md_files = [f for f in zf.namelist() if f.endswith(".md")]

            # 读取 & 下载md文件
            if md_files:
                md_file = next((f for f in md_files if "full" in f.lower()), md_files[0])
                with zf.open(md_file) as f:
                    md_content = f.read().decode("utf-8")
                    print(f"成功读取Markdown文件: {md_file}, 大小: {len(md_content) / 1024:.2f} KB")
                if md_content:
                    with open(md_output_path, "w", encoding="utf-8") as f:
                        f.write(md_content)

                    print(f"Markdown内容已保存到: {md_output_dir.resolve()}")
                    print(f"内容预览：{md_content[:500]}...")  # 预览前 500 字符
                    print("-" * 100)
            else:
                print("未找到Markdown文件")

            # 读取 & 下载图片 
            images = [f for f in zf.namelist() if f.lower().startswith("images/")]
            if images:
                image_output_dir = OUTPUT_DIR / input_file.stem / "images"
                image_output_dir.mkdir(parents=True, exist_ok=True)

                for img in images:
                    with zf.open(img) as f:
                        img_data = f.read()
                    if img_data:
                        with open(image_output_dir / f"{str(uuid.uuid4())}.jpg", "wb") as img_f:
                            img_f.write(img_data)
                    else:
                        print(f"警告：未能读取图片数据: {img}")
                print(f"成功下载并保存 {len(images)} 张图片到: {image_output_dir.resolve()}")
            else:
                print("未找到图片文件")

    else:
        print("未获取到解析结果的下载链接，无法继续")


start = time.perf_counter()
parsePDF()
cost_time = time.perf_counter() - start
print(f"PDF解析完成，耗时: {cost_time:.2f}秒")

正在获取上传URL...
获取上传URL成功，batch_id: 57dfaea8-a45a-4563-bf0e-4b6037d32bfb
正在上传PDF文件到MinerU的URL中...
PDF文件上传成功！
正在轮询检查处理状态...
解析完成，下载链接：https://cdn-mineru.openxlab.org.cn/pdf/2026-03-23/7f122dd8-139d-4403-a9e3-fdb94ddef3a2.zip
正在下载解析结果的zip文件，并解压...
成功读取Markdown文件: full.md, 大小: 46.97 KB
Markdown内容已保存到: D:\workspace\new-rag\python_backend\python_services\test\output\video
内容预览：# THE

# ARTOFASKING

# CHATGPT

# -FOR-

# HIGH-QUALITYANSWERS

A Complete Guide to Prompt Engineering Techniques

![](images/a450209c8e20c3edd09a88fef08b76944289083d546dad0d695e4980b4f844bb.jpg)

# The Art of Asking ChatGPT for High-Quality Answers

A Complete Guide to Prompt Engineering Techniques

Ibrahim John

Nzunda Technologies Limited

![](images/b92fc23b4fec06c87b42d45461ffd6cfb45f96425e565c236c69f6fb3c8c0bf7.jpg)

# Copyright $\circledcirc$ 2023 Ibrahim John

# All rights reserved

The...
----------------------------------------------------------------------------------------------------
成功下载并保存 2 张图片到: D:\work

In [44]:
# 解析后，切分文档
def split_md():
    with open(md_output_path, "r", encoding="utf-8") as f:
        md_content = f.read()
        print(f"Markdown内容预览：{md_content[:200]}...")  # 预览前 200 字符

    # 使用MULTILINE使^匹配每一行的开头
    has1 = bool(re.match(r"^#\s+", md_content, re.MULTILINE))
    has2 = bool(re.match(r"^##\s+", md_content, re.MULTILINE))

    if has1 and has2:
        md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=[("#", "header1"), ("##", "header2")])
        split_documents = md_splitter.split_text(md_content)
        print("md切分后的文档段落数:", len(split_documents))
        print("md切分后的文档段落预览:", split_documents[:2])
        print("-" * 100)
    else:
        recursive_splitter = RecursiveCharacterTextSplitter(
            separators=["\n\n", "\n"],
            chunk_size=400,
            chunk_overlap=50
        )
        split_documents = recursive_splitter.split_text(md_content)
        print("rt切分后的文档段落数:", len(split_documents))
        print("rt切分后的文档段落预览:", split_documents[:2])
        print("-" * 100)
    return split_documents


start = time.perf_counter()
split_documents = split_md()
cost_time = time.perf_counter() - start
print(f"文档切分完成，耗时: {cost_time:.2f}秒")

Markdown内容预览：# THE

# ARTOFASKING

# CHATGPT

# -FOR-

# HIGH-QUALITYANSWERS

A Complete Guide to Prompt Engineering Techniques

![](images/a450209c8e20c3edd09a88fef08b76944289083d546dad0d695e4980b4f844bb.jpg)

# ...
rt切分后的文档段落数: 152
rt切分后的文档段落预览: ['# THE\n\n# ARTOFASKING\n\n# CHATGPT\n\n# -FOR-\n\n# HIGH-QUALITYANSWERS\n\nA Complete Guide to Prompt Engineering Techniques\n\n![](images/a450209c8e20c3edd09a88fef08b76944289083d546dad0d695e4980b4f844bb.jpg)\n\n# The Art of Asking ChatGPT for High-Quality Answers\n\nA Complete Guide to Prompt Engineering Techniques\n\nIbrahim John\n\nNzunda Technologies Limited', 'Ibrahim John\n\nNzunda Technologies Limited\n\n![](images/b92fc23b4fec06c87b42d45461ffd6cfb45f96425e565c236c69f6fb3c8c0bf7.jpg)\n\n# Copyright $\\circledcirc$ 2023 Ibrahim John\n\n# All rights reserved\n\nThe characters and events portrayed in this book are fi ctitious. Any similarity to real persons, living or dead, is coincidental and not intended by the author.']
-------------

In [45]:
# 根据切分的chunk去构建data
datas = [StoredData(
    id=f"doc_{uuid.uuid4()}",
    chunk=chunk,
    sub_questions=[],
    subq_embeddings=[],
    summary="",
    summary_embedding=[],
    metadata={"source": pdf_name, }
) for i, chunk in enumerate(split_documents)]
print(f"构建数据对象完成，数据数量: {len(datas)}")
print(f"数据对象预览: {datas[0].model_dump_json(indent=2)}")

构建数据对象完成，数据数量: 152
数据对象预览: {
  "id": "doc_1f576e17-01b4-4675-8f28-388c5bac3dc5",
  "chunk": "# THE\n\n# ARTOFASKING\n\n# CHATGPT\n\n# -FOR-\n\n# HIGH-QUALITYANSWERS\n\nA Complete Guide to Prompt Engineering Techniques\n\n![](images/a450209c8e20c3edd09a88fef08b76944289083d546dad0d695e4980b4f844bb.jpg)\n\n# The Art of Asking ChatGPT for High-Quality Answers\n\nA Complete Guide to Prompt Engineering Techniques\n\nIbrahim John\n\nNzunda Technologies Limited",
  "sub_questions": [],
  "subq_embeddings": [],
  "summary": "",
  "summary_embedding": [],
  "metadata": {
    "source": "video.pdf"
  }
}


In [46]:
print(avg_len := sum(len(doc) for doc in split_documents) / len(split_documents))
print(max_len := max(len(doc) for doc in split_documents))
print(min_len := min(len(doc) for doc in split_documents))

327.7894736842105
541
21


In [47]:
ChatModel = ChatOpenAI(
    model_name=DEFAULT_MODEL,
    api_key=LITELLM_API_KEY,
    base_url=LITELLM_BASE_URL,
    max_retries=3,
    timeout=120,
)

EmbeddingModel = OpenAIEmbeddings(
    model=EMBEDDING_MODEL,
    api_key=LITELLM_API_KEY,
    base_url=LITELLM_BASE_URL,
)
print(f"已初始化ChatModel: {DEFAULT_MODEL} 和 EmbeddingModel: {EMBEDDING_MODEL}")

已初始化ChatModel: gpt-4o 和 EmbeddingModel: github_copilot/text-embedding-ada-002


In [48]:
from langchain_classic.output_parsers import OutputFixingParser
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel
import asyncio
from tqdm import tqdm
import os
from langchain_openai import ChatOpenAI
import time


class SubqAndSummary(BaseModel):
    subqs: list[str]
    summary: str

parser = PydanticOutputParser(pydantic_object=SubqAndSummary)
fixing_parser = OutputFixingParser.from_llm(parser=parser, llm=ChatModel)

# 创建 PromptTemplate
prompt_template = PromptTemplate.from_template(
    "你是一个专业的文档解析助手，负责为给定的文档段落生成子问题和摘要。\n"
    "请根据以下文档段落，生成3~5个相关的子问题和摘要。\n"
    "文档段落：{document_text}\n"
    "请严格按照以下JSON格式返回结果：{{'subqs':['subq1', 'subq2', ...], 'summary':'摘要内容'}}，请至少生成1条子问题"
)

# 添加超时参数
qwen = ChatOpenAI(
    model="qwen3.5-flash",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    timeout=180,  # 添加超时设置
    max_retries=2,  # 添加重试次数
)
print(f"已初始化ChatOpenAI模型: qwen3.5-flash，带超时和重试设置")

# 创建处理链
gen_chain = prompt_template | ChatModel | fixing_parser
print(f"已创建处理链")

subqs_path = OUTPUT_DIR / input_file.stem / "subqs.json"
os.makedirs(subqs_path.parent, exist_ok=True)
summaries_path = OUTPUT_DIR / input_file.stem / "summaries.json"
os.makedirs(summaries_path.parent, exist_ok=True)
print(f"输出路径已准备：\n  子问题: {subqs_path.resolve()}\n  摘要: {summaries_path.resolve()}")

已初始化ChatOpenAI模型: qwen3.5-flash，带超时和重试设置
已创建处理链
输出路径已准备：
  子问题: D:\workspace\new-rag\python_backend\python_services\test\output\video\subqs.json
  摘要: D:\workspace\new-rag\python_backend\python_services\test\output\video\summaries.json


In [49]:
import asyncio
import time

async def process_batch(chunk, batch_idx):
    inputs = [{"document_text": d[:3000]} for d in chunk]
    print(f"开始处理 batch {batch_idx}，包含 {len(chunk)} 个文档")
    t0 = time.time()
    try:
        results = await gen_chain.abatch(inputs)  # results 是列表
        t1 = time.time()
        print(f"batch {batch_idx} 完成，耗时 {t1-t0:.2f}s")
        # 返回每个文档的结果
        return {
            "idx": batch_idx,
            "results": [
                {"subqs": r.subqs, "summary": r.summary} for r in results
            ]
        }
    except Exception as e:
        t1 = time.time()
        print(f"batch {batch_idx} 异常: {e}，耗时 {t1-t0:.2f}s")
        # 返回空结果
        return {
            "idx": batch_idx,
            "results": [
                {"subqs": [], "summary": ""} for _ in chunk
            ]
        }

async def generate_batches_async_concurrent(datas, batch_size=16, max_concurrency=4):
    docs = [d.chunk for d in datas]
    batches = [docs[i:i+batch_size] for i in range(0, len(docs), batch_size)]
    sem = asyncio.Semaphore(max_concurrency)

    async def sem_task(chunk, batch_idx):
        async with sem:
            return await process_batch(chunk, batch_idx)

    tasks = [sem_task(chunk, idx) for idx, chunk in enumerate(batches)]
    t0 = time.time()
    results_list = await asyncio.gather(*tasks)
    t1 = time.time()
    print(f"全部批次处理完成，总耗时 {t1-t0:.2f}s")

    for results in results_list:
        chunk_id = results.get("idx")
        batch_start = chunk_id * batch_size
        batch_end = min(batch_start + batch_size, len(datas))
        for i, idx in enumerate(range(batch_start, batch_end)):
            doc_result = results["results"][i]
            datas[idx].sub_questions = [q for q in doc_result["subqs"] if q]
            datas[idx].summary = doc_result["summary"]

# 在 notebook 里用 top-level await
await generate_batches_async_concurrent(datas, batch_size=16, max_concurrency=8)

开始处理 batch 0，包含 16 个文档
开始处理 batch 1，包含 16 个文档
开始处理 batch 2，包含 16 个文档
开始处理 batch 3，包含 16 个文档
开始处理 batch 4，包含 16 个文档
开始处理 batch 5，包含 16 个文档
开始处理 batch 6，包含 16 个文档
开始处理 batch 7，包含 16 个文档
batch 1 完成，耗时 18.59s
开始处理 batch 8，包含 16 个文档
batch 0 完成，耗时 21.50s
开始处理 batch 9，包含 8 个文档
batch 4 完成，耗时 24.44s
batch 3 完成，耗时 24.82s
batch 5 完成，耗时 25.82s
batch 6 完成，耗时 27.27s
batch 7 完成，耗时 27.37s
batch 2 完成，耗时 30.63s
batch 9 完成，耗时 9.76s
batch 8 完成，耗时 15.05s
全部批次处理完成，总耗时 33.65s


In [50]:
datas_path = OUTPUT_DIR / input_file.stem / "datas.json"
with open(datas_path, "w", encoding="utf-8") as f:
    json.dump([d.model_dump() for d in datas], f, ensure_ascii=False, indent=2)
print(f"完整数据对象已保存到: {datas_path.resolve()}")

完整数据对象已保存到: D:\workspace\new-rag\python_backend\python_services\test\output\video\datas.json


In [51]:
print(datas[50].model_dump_json(indent=2))

with open(subqs_path, "w", encoding="utf-8") as f:
    json.dump([d.sub_questions for d in datas], f, ensure_ascii=False, indent=2)
with open(summaries_path, "w", encoding="utf-8") as f:
    json.dump([d.summary for d in datas], f, ensure_ascii=False, indent=2)
print(f"子问题和摘要已保存到:\n  {subqs_path.resolve()}\n  {summaries_path.resolve()}")

{
  "id": "doc_723a8a97-4d88-4102-9ff4-266ed862761b",
  "chunk": "Example 2: Text Summarization\n\nTask: Summarize a news article   \nInstructions: The summary should be consistent with the information provided in the article   \nPrompt formula: \"Summarize the following news article in a way that is consistent with the information provided [insert news article]\"\n\nExample 3: Text Completion",
  "sub_questions": [
    "What is the main purpose of the task in Example 2: Text Summarization?",
    "What are the key instructions provided for summarizing a news article in Example 2?",
    "How does the prompt formula guide the summarization process in Example 2?"
  ],
  "subq_embeddings": [],
  "summary": "Example 2 describes the task of summarizing a news article with clear instructions to ensure the summary remains consistent with the article. A specific prompt formula is provided to guide the summarization process.",
  "summary_embedding": [],
  "metadata": {
    "source": "video.pdf"


In [54]:
import math

async def batch_embed_texts(texts, batch_size=64):
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        batch_embeds = await EmbeddingModel.aembed_documents(batch)
        embeddings.extend(batch_embeds)
    return embeddings

async def generate_and_fill_embeddings(datas):
    print("正在生成并填充摘要和子问题的嵌入...")
    valid_indices = [i for i, d in enumerate(datas) if d.summary and d.sub_questions]
    valid_datas = [datas[i] for i in valid_indices]
    print(len(valid_datas), "条数据需要生成嵌入")
    summary_texts = [d.summary for d in valid_datas]
    subq_texts = [subq for d in valid_datas for subq in d.sub_questions]
    # 分批生成 embedding
    summary_embeddings = await batch_embed_texts(summary_texts, batch_size=32)
    subq_embeddings = await batch_embed_texts(subq_texts, batch_size=64)
    subq_offset = 0
    for idx, d in zip(valid_indices, valid_datas):
        datas[idx].summary_embedding = summary_embeddings[valid_datas.index(d)]
        datas[idx].subq_embeddings = subq_embeddings[subq_offset:subq_offset + len(d.sub_questions)]
        subq_offset += len(d.sub_questions)

await generate_and_fill_embeddings(datas)

datas_path = OUTPUT_DIR / input_file.stem / "full_datas.json"
with open(datas_path, "w", encoding="utf-8") as f:
    json.dump([d.model_dump() for d in datas], f, ensure_ascii=False, indent=2)
print(f"完整数据对象已保存到: {datas_path.resolve()}")

正在生成并填充摘要和子问题的嵌入...
152 条数据需要生成嵌入
完整数据对象已保存到: D:\workspace\new-rag\python_backend\python_services\test\output\video\full_datas.json


In [ ]:
# 信号量优化+gather并行---对embedding做性能优化


In [64]:
print(len(datas[0].summary_embedding))

1536


In [92]:
from typing import Optional
from pymilvus import connections, Collection, FieldSchema, CollectionSchema, DataType, list_collections
import time

class MilvusImporter:
    def __init__(self, host="localhost", port="19530"):
        self.host = host
        self.port = port
        self.summaries_collection = None
        self.subquestions_collection = None

    def connect(self):
        try:
            connections.connect(
                alias="default",
                host=self.host,
                port=self.port
            )
            print("Milvus连接成功")
            return True
        except Exception as e:
            print(f"Milvus连接失败: {e}")
            return False

    def get_collections(self):
        try:
            return list_collections()
        except Exception as e:
            print(f"获取集合列表失败: {e}")
            return []

    def create_collections(self):
        try:
            # 创建摘要集合
            summaries_fields = [
                FieldSchema(name="chunk_id", dtype=DataType.INT64, is_primary=True, auto_id=False),
                FieldSchema(name="chunk_text", dtype=DataType.VARCHAR, max_length=65535),
                FieldSchema(name="summary_text", dtype=DataType.VARCHAR, max_length=2000),
                FieldSchema(name="summary_vector", dtype=DataType.FLOAT_VECTOR, dim=1536),
                FieldSchema(name="created_at", dtype=DataType.INT64)
            ]
            summaries_schema = CollectionSchema(fields=summaries_fields, description="文档摘要集合")

            collections = self.get_collections()
            if "chunk_summaries" in collections:
                Collection("chunk_summaries").drop()
                print("已删除旧的摘要集合")

            self.summaries_collection = Collection(name="chunk_summaries", schema=summaries_schema)

            # 创建摘要向量索引
            summaries_index_params = {
                "index_type": "AUTOINDEX",
                "metric_type": "COSINE"
            }
            self.summaries_collection.create_index(field_name="summary_vector", index_params=summaries_index_params)
            print("摘要集合创建成功")

            # 创建子问题集合
            subquestions_fields = [
                FieldSchema(name="subquestion_id", dtype=DataType.INT64, is_primary=True, auto_id=True),
                FieldSchema(name="chunk_id", dtype=DataType.INT64),
                FieldSchema(name="chunk_text", dtype=DataType.VARCHAR, max_length=65535),
                FieldSchema(name="question_text", dtype=DataType.VARCHAR, max_length=500),
                FieldSchema(name="question_vector", dtype=DataType.FLOAT_VECTOR, dim=1536),
                FieldSchema(name="created_at", dtype=DataType.INT64)
            ]
            subquestions_schema = CollectionSchema(fields=subquestions_fields, description="文档子问题集合")

            if "chunk_subquestions" in collections:
                Collection("chunk_subquestions").drop()
                print("已删除旧的子问题集合")

            self.subquestions_collection = Collection(name="chunk_subquestions", schema=subquestions_schema)

            # 创建子问题向量索引
            subquestions_index_params = {
                "index_type": "AUTOINDEX",
                "metric_type": "COSINE"
            }
            self.subquestions_collection.create_index(field_name="question_vector",
                                                      index_params=subquestions_index_params)
            print("子问题集合创建成功")

            return True
        except Exception as e:
            print(f"创建集合失败: {e}")
            return False

    def import_data(self, datas):
        if not self.summaries_collection or not self.subquestions_collection:
            print("错误：集合未创建")
            return False

        print(f"开始导入 {len(datas)} 个文档数据")

        summaries_chunk_ids = []
        summaries_chunk_texts = []
        summaries_texts = []
        summaries_vectors = []
        summaries_created_at = []

        subquestions_chunk_ids = []
        subquestions_chunk_texts = []
        subquestions_texts = []
        subquestions_vectors = []
        subquestions_created_at = []

        timestamp = int(time.time() * 1000)

        for i, doc in enumerate(datas):
            chunk_id = i + 1
            if doc.summary and doc.summary_embedding is not None:
                summaries_chunk_ids.append(chunk_id)
                summaries_chunk_texts.append(doc.chunk)
                summaries_texts.append(doc.summary)
                summaries_vectors.append(doc.summary_embedding)
                summaries_created_at.append(timestamp)
            if doc.sub_questions and doc.subq_embeddings:
                for subq, embedding in zip(doc.sub_questions, doc.subq_embeddings):
                    if subq and embedding is not None:
                        subquestions_chunk_ids.append(chunk_id)
                        subquestions_chunk_texts.append(doc.chunk)
                        subquestions_texts.append(subq)
                        subquestions_vectors.append(embedding)
                        subquestions_created_at.append(timestamp)

        if summaries_chunk_ids:
            print(f"导入 {len(summaries_chunk_ids)} 条摘要数据")
            summaries_entities = [
                summaries_chunk_ids,
                summaries_chunk_texts,
                summaries_texts,
                summaries_vectors,
                summaries_created_at
            ]
            try:
                self.summaries_collection.insert(summaries_entities)
                print("摘要数据导入成功")
            except Exception as e:
                print(f"导入摘要数据失败: {e}")
                return False

        if subquestions_chunk_ids:
            print(f"导入 {len(subquestions_chunk_ids)} 条子问题数据")
            subquestions_entities = [
                subquestions_chunk_ids,
                subquestions_chunk_texts,
                subquestions_texts,
                subquestions_vectors,
                subquestions_created_at
            ]
            try:
                self.subquestions_collection.insert(subquestions_entities)
                print("子问题数据导入成功")
            except Exception as e:
                print(f"导入子问题数据失败: {e}")
                return False

        self.summaries_collection.load()
        self.subquestions_collection.load()
        print("数据导入完成")
        return True
    
    def query(
            self,
            query_embedding: list,
            search_params: Optional[dict],
            limit: int = 5,
            output_fields_subq=None,
            output_fields_summary=None,
            expr: Optional[str] = None,
    ):
        if output_fields_subq is None:
            output_fields_subq = ["chunk_id", "summary_text", "created_at"]
        if output_fields_summary is None:
            output_fields_summary = ["chunk_text", "question_text", "created_at"]
        res1 = self.summaries_collection.search(
            data=query_embedding,
            anns_field="summary_vector",
            param=search_params,
            limit=limit,
            expr=expr,
            output_fields=output_fields_subq
        )
        
        res2 = self.subquestions_collection.search(
            data=query_embedding,
            anns_field="question_vector",
            param=search_params,
            limit=limit,
            expr=expr,
            output_fields=output_fields_summary
        )
        return res1+res2
    
    def verify_import(self):
        if not self.summaries_collection or not self.subquestions_collection:
            print("错误：集合未创建")
            return False

        try:
            summaries_count = self.summaries_collection.num_entities
            print(f"摘要集合行数: {summaries_count}")

            subquestions_count = self.subquestions_collection.num_entities
            print(f"子问题集合行数: {subquestions_count}")

            if summaries_count > 0:
                result = self.summaries_collection.query(
                    expr="chunk_id > 0",
                    limit=1,
                    output_fields=["chunk_id", "summary_text"]
                )
                if result:
                    print(f"摘要数据示例: {result[0]}")

            if subquestions_count > 0:
                result = self.subquestions_collection.query(
                    expr="chunk_id > 0",
                    limit=1,
                    output_fields=["chunk_id", "question_text"]
                )
                if result:
                    print(f"子问题数据示例: {result[0]}")

            return True
        except Exception as e:
            print(f"验证导入结果失败: {e}")
            return False

    def close(self):
        try:
            connections.disconnect("default")
            print("Milvus连接已关闭")
        except Exception as e:
            print(f"关闭连接失败: {e}")
            
print("导入类定义")

导入类定义


In [97]:
# 初始化导入器
importer = MilvusImporter()

# 连接到Milvus
if importer.connect():
    # 创建集合
    if importer.create_collections():
        # 导入数据（使用您已经对齐的datas）
        if importer.import_data(datas):
            # 验证导入结果
            importer.verify_import()
        else:
            print("数据导入失败")
    else:
        print("创建集合失败")
    
else:
    print("连接Milvus失败")

Milvus连接成功
已删除旧的摘要集合
摘要集合创建成功
已删除旧的子问题集合
子问题集合创建成功
开始导入 152 个文档数据
导入 152 条摘要数据
摘要数据导入成功
导入 546 条子问题数据
子问题数据导入成功
数据导入完成
摘要集合行数: 0
子问题集合行数: 0


In [98]:
# 测试查询
query = "What is prompt engineering"
query_embedding = [EmbeddingModel.embed_query(query)]
# print(query_embedding)
expr = "chunk_id > 100"

search_params = {
    "metric_type": "COSINE",
    "params": {"nprobe": 10}
}

results = importer.query( query_embedding=query_embedding, search_params=search_params)
print(results)

# for hits in results:
#     for hit in hits:
#         print(hit.entity.get("chunk_id"), hit.distance, hit.entity.get("summary_text"))


[[{'chunk_id': 4, 'distance': 0.6672012805938721, 'entity': {'summary_text': "This document serves as an introduction to various prompt engineering techniques and their applications. Key areas include Instruction Prompt Techniques, Role Prompting, Zero/One/Few Shot Prompting, the 'Let’s think about this' method, and Self-Consistency Prompting, which are designed to enhance and refine AI interaction.", 'created_at': 1774276881179, 'chunk_id': 4}}, {'chunk_id': 16, 'distance': 0.6670522093772888, 'entity': {'summary_text': "The passage focuses on defining key elements of prompt engineering, including the task, instructions, and the model's role. It emphasizes their importance in guiding ChatGPT and introduces the exploration of various techniques for enhancing prompt design.", 'created_at': 1774276881179, 'chunk_id': 16}}, {'chunk_id': 11, 'distance': 0.6166517734527588, 'entity': {'summary_text': 'This text introduces the book and its focus on prompt engineering techniques, offering exa

In [95]:
# 关闭连接
importer.close()

Milvus连接已关闭
